# 6. Reuse in PCAIR

When solving sequences of linear systems — such as in time-stepping, nonlinear iterations, or eigenvalue problems — rebuilding the full PCAIR hierarchy from scratch for each solve is expensive, particularly in parallel.

PFLARE provides several reuse mechanisms that store intermediate data from one setup and reuse it in the next, reducing setup cost significantly; please see the documentation **[docs/reuse.md](../docs/reuse.md)** for more details.

This notebook covers some aspects of reuse with PCAIR:
1. **Reuse sparsity** — store the CF splitting, SpGEMM results, and operators to accelerate subsequent setups.
2. **Reuse amount** — control the memory/speed trade-off by choosing how much data to store.

All examples solve a sequence of two linear systems where the matrix entries change between solves but the sparsity pattern remains the same.

## Memory in reduction multigrids

Classical AMG methods typically apply smoothers that update all unknowns — both fine and coarse — and so must keep the full coarse-grid matrix alive at every level for use in both the setup and the solve. 

The reduction multigrid in PFLARE, on the other hand, use **F-point-only smoothing by default**: on each level the smoother only acts on the F-points, and the residual is restricted to the C-points to form the next level. 

This means the only matrices needed during the solve phase are $A_{ff}$ (to apply the F-point smoother) and $A_{fc}$ (to compute the F-point residual after a C-point correction). The coarse matrix $A$ from the previous level is not needed once the extraction of $A_{ff}$, $A_{fc}$, $A_{cf}$, and the new coarse matrix $A_c$ is complete. 

PFLARE therefore **destroys the coarse matrix on each level after the setup**, keeping only $A_{ff}$, $A_{fc}$, the restriction and prolongation operators, and the approximate inverse $A_{ff}^{-1}$. This makes the memory footprint of a single PCAIR solve considerably lower than it would be in a classical AMG that retained the full hierarchy of coarse matrices.

Reuse changes this picture. To reuse the sparsity structure on the next setup, we need to store some of the matrices we normally destroy in the setup. 

An alternative design would be to store only the coarse matrix on each level and re-extract A_ff/A_fc on the fly during the solve (or equivalent). This would save memory but would significantly slow down every solve, since submatrix extraction is far more expensive than a direct SpMV against a pre-assembled submatrix. 

The philosophy in PFLARE is to target the **lowest memory and fastest solve** for a single-solve use case, accepting that reuse carries a memory cost.

In [ ]:
import sys
import os
import tempfile
import time
import numpy as np

import petsc4py
petsc4py.init(sys.argv)
from petsc4py import PETSc

import pflare

sys.path.insert(0, os.path.normpath(os.path.join(os.getcwd(), '..', 'tools')))
from parse_pflare_output import parse_pflare_output

print("Ready.")


## 1. Test problem and helpers

We use a 2D five-point Laplacian on an $N \times N$ grid. Between the first and second solve we perturb the diagonal entries so that the matrix changes but retains the same sparsity pattern — mimicking a time-stepping or nonlinear iteration scenario.

We also build a helper that captures the PCAIR stats output (using the same approach as notebook 05) so we can extract the **storage complexity** and **reuse storage complexity** alongside the iteration count.

In [ ]:
def build_2d_laplacian(N):
    """Build the standard 2D five-point Laplacian on an N x N grid."""
    n = N * N
    A = PETSc.Mat().create(PETSc.COMM_WORLD)
    A.setSizes([n, n])
    A.setFromOptions()
    A.setUp()

    Istart, Iend = A.getOwnershipRange()
    for II in range(Istart, Iend):
        i = II // N
        j = II - i * N
        if i > 0:     A.setValue(II, II - N, -1.0)
        if i < N - 1: A.setValue(II, II + N, -1.0)
        if j > 0:     A.setValue(II, II - 1, -1.0)
        if j < N - 1: A.setValue(II, II + 1, -1.0)
        A.setValue(II, II, 4.0)

    A.assemblyBegin()
    A.assemblyEnd()
    return A


def perturb_diagonal(A, delta):
    """Add delta to every diagonal entry of A (same sparsity, different values)."""
    Istart, Iend = A.getOwnershipRange()
    for II in range(Istart, Iend):
        A.setValue(II, II, delta, PETSc.InsertMode.ADD_VALUES)
    A.assemblyBegin()
    A.assemblyEnd()


def run_pcair_two_solves(N, configure_pc=None, delta=0.5, rtol=1e-8, max_it=100):
    """
    Build a Laplacian, do two solves (perturbing diagonal by delta between them),
    and return [(iters, complexities), ...] where complexities is the dict from
    parse_pflare_output (keys: grid, operator, cycle, storage, reuse_storage).
    PCAIR stats are captured by redirecting C-level stdout.
    """
    A = build_2d_laplacian(N)
    b = A.createVecRight()
    b.set(1.0)

    ksp = PETSc.KSP().create()
    ksp.setType(PETSc.KSP.Type.GMRES)
    ksp.setTolerances(rtol=rtol, max_it=max_it)
    pc = ksp.getPC()
    pc.setType("air")
    pflare.pcair_set_print_stats_timings(pc, True)
    if configure_pc is not None:
        configure_pc(pc)
    ksp.setFromOptions()

    results = []
    for solve_idx in range(2):
        if solve_idx == 1:
            perturb_diagonal(A, delta)

        # Capture C-level stdout to parse stats
        sys.stdout.flush()
        old_fd = os.dup(1)
        r_fd, w_fd = os.pipe()
        os.dup2(w_fd, 1)
        os.close(w_fd)
        try:
            x = b.duplicate()
            x.set(0.0)
            ksp.setOperators(A, A)
            ksp.solve(b, x)
            iters = ksp.getIterationNumber()
            x.destroy()
        finally:
            sys.stdout.flush()
            os.dup2(old_fd, 1)
            os.close(old_fd)

        chunks = []
        while True:
            chunk = os.read(r_fd, 65536)
            if not chunk:
                break
            chunks.append(chunk)
        os.close(r_fd)
        captured = b''.join(chunks).decode('utf-8', errors='replace')

        fd, tmpfile = tempfile.mkstemp(suffix='.txt')
        try:
            with os.fdopen(fd, 'w') as fh:
                fh.write(captured)
            data = parse_pflare_output(tmpfile)
        finally:
            os.unlink(tmpfile)

        results.append((iters, data['complexities']))

    ksp.destroy()
    A.destroy()
    b.destroy()
    return results


N = 40
print(f"Problem size: {N}x{N} = {N*N} unknowns")


## 2. Reuse sparsity

With `-pc_air_reuse_sparsity` enabled, PCAIR stores the CF splitting, repartitioning index sets, and all intermediate SpGEMM results from the first setup. The second setup then reuses this data, avoiding the symbolic matrix products and CF splitting.

We compare a full rebuild (no reuse) against reuse with the default `reuse_amount = 3` (store everything). The **storage complexity** measures the total non-zeros stored for the solve (A_ff, A_fc, inv_A_ff, prolongators, restrictors, coarse matrices), normalised by the non-zeros in the top-level matrix. 

The **reuse storage complexity** measures the additional non-zeros stored exclusively for reuse (the intermediate SpGEMM matrices), again normalised by the top-level non-zeros.

In [ ]:
# No reuse
res_no_reuse = run_pcair_two_solves(N, configure_pc=None)

# With reuse sparsity (default reuse_amount=3)
def cfg_reuse(pc):
    pflare.pcair_set_reuse_sparsity(pc, True)

res_reuse = run_pcair_two_solves(N, configure_pc=cfg_reuse)

print(f"{'':30}  {'iters':>6}  {'storage_cplx':>13}  {'reuse_storage_cplx':>19}")
for label, results in [("No reuse", res_no_reuse), ("Reuse sparsity (amt=3)", res_reuse)]:
    for solve_idx, (iters, stats) in enumerate(results):
        sc  = f"{stats['storage']:.3f}"       if stats['storage']       is not None else "n/a"
        rsc = f"{stats['reuse_storage']:.3f}" if stats['reuse_storage'] is not None else "n/a"
        tag = f"{label} — solve {solve_idx+1}"
        print(f"  {tag:30}  {iters:>6}  {sc:>13}  {rsc:>19}")

The reuse storage complexity shows the memory overhead of storing the intermediate SpGEMM matrices for reuse. This is on top of the storage complexity of the solve itself, which is broadly similar with and without reuse.

## 3. Controlling memory with `reuse_amount`

When `-pc_air_reuse_sparsity` is enabled, `-pc_air_reuse_amount` controls how much intermediate data is stored between setups:

| `-pc_air_reuse_amount` | Data stored | Setup cost (2nd solve) |
|:-:|---|:-:|
| **1** | CF splitting only (IS_fine, IS_coarse) | Full rebuild — all SpGEMMs and operator extractions repeated |
| **2** | CF splitting + SpGEMM sparsity patterns (W, Z, AP, RAP, RAP_DROP, ...) | SpGEMM sparsity reused; A_ff/A_fc/prolongators/restrictors rebuilt |
| **3** | Everything including approximate inverses and grid-transfer operators | Fastest possible second setup (default) |

The reuse storage complexity printed below reflects exactly what extra memory each amount requires.

In [ ]:
print(f"{'reuse_amount':>13}  {'solve':>6}  {'iters':>6}  {'storage_cplx':>13}  {'reuse_storage_cplx':>19}")
for amount in [1, 2, 3]:
    def cfg_amount(pc, amt=amount):
        pflare.pcair_set_reuse_sparsity(pc, True)
        pflare.pcair_set_reuse_amount(pc, amt)

    results = run_pcair_two_solves(N, configure_pc=cfg_amount)
    for solve_idx, (iters, stats) in enumerate(results):
        sc  = f"{stats['storage']:.3f}"       if stats['storage']       is not None else "n/a"
        rsc = f"{stats['reuse_storage']:.3f}" if stats['reuse_storage'] is not None else "n/a"
        print(f"  {amount:>13}  {solve_idx+1:>6}  {iters:>6}  {sc:>13}  {rsc:>19}")
    print()

As expected:
- `reuse_amount = 1` has zero reuse storage complexity in serial (only the CF index sets are stored, which are negligible).
- `reuse_amount = 2` stores the SpGEMM sparsity patterns — visible as a non-zero reuse storage complexity, but lower than amount 3.
- `reuse_amount = 3` stores everything, giving the highest reuse storage complexity but the fastest second setup.

## Summary

PCAIR provides the ability to store intermediate data to speed up the setup during multiple solves, at the cost of additional memory use:

- **Reuse sparsity** (`-pc_air_reuse_sparsity`): Store intermediate data to reuse the CF splitting and SpGEMM symbolic products. Gives the largest speedup in setup time, particularly in parallel.

- **Reuse amount** (`-pc_air_reuse_amount`): Control the amount of intermediate data stored. Amount 1 stores only the CF splitting (minimal extra memory), amount 2 additionally stores SpGEMM sparsity patterns (moderate overhead), amount 3 stores everything including approximate inverses (highest overhead, fastest second setup; default).